# Processamento em lote — Equilíbrio / SDA (FP + ACC)

Este notebook permite carregar **vários arquivos `.mat`** no mesmo formato de `Ariatacho balance OA1.mat` e processa automaticamente:

- variáveis globais de FP e ACC;
- SDA em AP, ML e R;
- Critical Point (CP);
- Slope OL e Slope CL;
- OL Sway;
- média, desvio-padrão, mediana, mínimo e máximo de todos os arquivos;
- exportação para Excel.

### Correções incorporadas em relação ao notebook original
1. O resultante `R` é usado diretamente, sem aplicar uma segunda raiz quadrada.
2. O CP/Sway resultante do ACC usa `tempo2_accR` e `MSDseries_accR`, e não os arrays da FP.
3. O output de Slope OL ML do ACC usa a variável do ACC.
4. O CP continua seguindo **o mesmo critério do notebook original**: primeiro ponto em que a derivada do MSD se torna negativa.
5. O arquivo de resultados inclui o nome do arquivo para rastreabilidade.

> Estrutura esperada do `.mat`: variável `dataToExport`, com FP em `[0,0]` (`AP`, `ML`, `Tempo`) e ACC em `[0,3]` (`accAP`, `accML`, `Tempo`).


In [ ]:

import os
import numpy as np
import pandas as pd
import scipy.io
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
from scipy.spatial import ConvexHull
from IPython.display import display

FS = 100.0
FP_LOWCUT = 0.1
FP_HIGHCUT = 4.0
ACC_LOWCUT = 0.1
ACC_HIGHCUT = 3.0
FILTER_ORDER = 2


In [ ]:

def _flatten_mat_field(field):
    arr = np.asarray(field)
    while arr.dtype == object and arr.size == 1:
        arr = np.asarray(arr.flat[0])
    return np.asarray(arr).squeeze().astype(float)


def bandpass(signal, lowcut, highcut, fs=FS, order=FILTER_ORDER):
    signal = np.asarray(signal, dtype=float)
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype="band")
    return filtfilt(b, a, signal)


def ellipse_metrics(x, y, confidence=0.95):
    points = np.column_stack((x, y))
    hull = ConvexHull(points)
    boundary = points[hull.vertices]
    cov = np.cov(boundary, rowvar=False)

    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.maximum(eigvals, 0)
    eigvals = np.sort(eigvals)[::-1]

    scale = np.sqrt(-2 * np.log(1 - confidence)) / 2
    axes = np.sqrt(eigvals) * scale
    major, minor = float(np.max(axes)), float(np.min(axes))
    area = float(np.pi * major * minor)
    return area, major, minor


def global_metrics(ap, ml):
    ap = np.asarray(ap, dtype=float)
    ml = np.asarray(ml, dtype=float)
    r = np.sqrt(ap**2 + ml**2)

    # Mantém também a fórmula original do notebook para auditoria.
    total_original = float(np.sum(np.sqrt(r)))

    # Comprimento de trajetória entre amostras consecutivas.
    total_path = float(np.sum(np.sqrt(np.diff(ap)**2 + np.diff(ml)**2)))

    rms_ap = float(np.sqrt(np.mean(ap**2)))
    rms_ml = float(np.sqrt(np.mean(ml**2)))
    area, major, minor = ellipse_metrics(ml, ap)

    return {
        "Total_original": total_original,
        "Total_path": total_path,
        "RMS_AP": rms_ap,
        "RMS_ML": rms_ml,
        "Ellipse_area": area,
        "Ellipse_major_axis": major,
        "Ellipse_minor_axis": minor,
    }


def msd_curve(series, fs=FS):
    series = np.asarray(series, dtype=float)
    n = len(series)
    max_step = n // 2

    msd = np.empty(max_step, dtype=float)
    lags = np.arange(1, max_step + 1, dtype=float) / fs

    for dt in range(1, max_step + 1):
        delta = series[dt:] - series[:-dt]
        msd[dt - 1] = np.mean(delta**2)

    return lags, msd


def sda_metrics(series, fs=FS):
    """
    Reproduz o critério de CP do notebook original:
    CP = primeiro lag em que diff(MSD) < 0.
    """
    lags, msd = msd_curve(series, fs)
    deriv = np.diff(msd)
    neg = np.flatnonzero(deriv < 0)

    if len(neg) == 0:
        return {"CP_s": np.nan, "Slope_OL": np.nan,
                "OL_Sway": np.nan, "Slope_CL": np.nan}

    h1 = int(neg[0])
    cp = float(lags[h1])
    sway = float(msd[h1])

    slope_ol = np.nan
    slope_cl = np.nan

    if h1 >= 2:
        slope_ol = float(np.polyfit(lags[:h1], msd[:h1], 1)[0])

    if len(lags[h1:-1]) >= 2:
        slope_cl = float(np.polyfit(lags[h1:-1], msd[h1:-1], 1)[0])

    return {"CP_s": cp, "Slope_OL": slope_ol,
            "OL_Sway": sway, "Slope_CL": slope_cl}


def extract_mat_signals(mat_path):
    data = scipy.io.loadmat(mat_path)
    if "dataToExport" not in data:
        raise ValueError("Arquivo sem a variável dataToExport.")

    arr = data["dataToExport"]

    # Force platform
    fp_struct = arr[0, 0]
    fp_ap = _flatten_mat_field(fp_struct["AP"])
    fp_ml = _flatten_mat_field(fp_struct["ML"])

    nfp = min(len(fp_ap), len(fp_ml), 4501)
    fp_ap = fp_ap[:nfp]
    fp_ml = fp_ml[:nfp]

    fp_ap = bandpass(fp_ap, FP_LOWCUT, FP_HIGHCUT)
    fp_ml = bandpass(fp_ml, FP_LOWCUT, FP_HIGHCUT)

    # Accelerometer
    acc_struct = arr[0, 3]
    acc_ap = _flatten_mat_field(acc_struct["accAP"])
    acc_ml = _flatten_mat_field(acc_struct["accML"])
    acc_t = _flatten_mat_field(acc_struct["Tempo"])

    n = min(len(acc_ap), len(acc_ml), len(acc_t))
    acc_ap, acc_ml, acc_t = acc_ap[:n], acc_ml[:n], acc_t[:n]

    order = np.argsort(acc_t)
    acc_t, acc_ap, acc_ml = acc_t[order], acc_ap[order], acc_ml[order]

    unique_t, unique_idx = np.unique(acc_t, return_index=True)
    acc_t = unique_t
    acc_ap = acc_ap[unique_idx]
    acc_ml = acc_ml[unique_idx]

    t_new = np.arange(acc_t[0], acc_t[-1], 1/FS)
    acc_ap = interp1d(acc_t, acc_ap, kind="linear",
                      bounds_error=False, fill_value="extrapolate")(t_new)
    acc_ml = interp1d(acc_t, acc_ml, kind="linear",
                      bounds_error=False, fill_value="extrapolate")(t_new)

    acc_ap = bandpass(acc_ap, ACC_LOWCUT, ACC_HIGHCUT)
    acc_ml = -1.0 * bandpass(acc_ml, ACC_LOWCUT, ACC_HIGHCUT)

    return fp_ap, fp_ml, acc_ap, acc_ml


def process_file(mat_path):
    fp_ap, fp_ml, acc_ap, acc_ml = extract_mat_signals(mat_path)

    # CORRIGIDO: R é usado diretamente.
    fp_r = np.sqrt(fp_ap**2 + fp_ml**2)
    acc_r = np.sqrt(acc_ap**2 + acc_ml**2)

    result = {"Arquivo": os.path.basename(mat_path)}

    for device, ap, ml in [("FP", fp_ap, fp_ml),
                           ("ACC", acc_ap, acc_ml)]:
        gm = global_metrics(ap, ml)
        for k, v in gm.items():
            result[f"{device}_{k}"] = v

    for device, signals in [
        ("FP", {"AP": fp_ap, "ML": fp_ml, "R": fp_r}),
        ("ACC", {"AP": acc_ap, "ML": acc_ml, "R": acc_r}),
    ]:
        for axis, sig in signals.items():
            sm = sda_metrics(sig)
            for k, v in sm.items():
                result[f"{device}_{axis}_{k}"] = v

    return result


## 1. Carregar os arquivos

In [ ]:

# === Google Colab: carregue vários .mat de uma vez ===
try:
    from google.colab import files
    uploaded = files.upload()
    mat_files = []
    for filename, content in uploaded.items():
        if filename.lower().endswith(".mat"):
            with open(filename, "wb") as f:
                f.write(content)
            mat_files.append(filename)
except ImportError:
    mat_files = []

# === Rodando localmente: coloque os .mat na pasta abaixo ===
PASTA_LOCAL = "."

if not mat_files:
    mat_files = [
        os.path.join(PASTA_LOCAL, f)
        for f in os.listdir(PASTA_LOCAL)
        if f.lower().endswith(".mat")
    ]

print(f"{len(mat_files)} arquivo(s) .mat encontrado(s).")
for f in mat_files:
    print(" -", f)


## 2. Processar todos os arquivos

In [ ]:

results = []
errors = []

for path in mat_files:
    try:
        results.append(process_file(path))
        print("OK:", os.path.basename(path))
    except Exception as e:
        errors.append({"Arquivo": os.path.basename(path), "Erro": str(e)})
        print("ERRO:", os.path.basename(path), "->", e)

df_individual = pd.DataFrame(results)

if len(df_individual) == 0:
    raise RuntimeError("Nenhum arquivo foi processado com sucesso.")

display(df_individual)


## 3. Médias, DP e Critical Points

In [ ]:

numeric_cols = df_individual.select_dtypes(include=[np.number]).columns

summary_rows = []
for col in numeric_cols:
    vals = pd.to_numeric(df_individual[col], errors="coerce")
    summary_rows.append({
        "Variavel": col,
        "N": int(vals.notna().sum()),
        "Media": vals.mean(),
        "DP": vals.std(ddof=1),
        "Mediana": vals.median(),
        "Min": vals.min(),
        "Max": vals.max(),
    })

df_summary = pd.DataFrame(summary_rows)

cp_cols = [c for c in numeric_cols if c.endswith("_CP_s")]
df_cp = df_individual[["Arquivo"] + cp_cols].copy()

print("Critical Points por arquivo:")
display(df_cp)

print("Resumo dos Critical Points:")
display(df_summary[df_summary["Variavel"].isin(cp_cols)])


## 4. Exportar Excel

In [ ]:

output = "resultados_balance_sda_lote.xlsx"

with pd.ExcelWriter(output, engine="openpyxl") as writer:
    df_individual.to_excel(writer, sheet_name="Resultados_individuais", index=False)
    df_cp.to_excel(writer, sheet_name="Critical_Points", index=False)
    df_summary.to_excel(writer, sheet_name="Resumo_media_DP", index=False)
    if errors:
        pd.DataFrame(errors).to_excel(writer, sheet_name="Erros", index=False)

print("Arquivo gerado:", output)

try:
    from google.colab import files
    files.download(output)
except ImportError:
    pass
